# Supplementary figure: depth selectivity of the V1 population

Panels **A-I**: three further example depth-tuned neurons (near, middle and far
preferred depth) from the Fig. 1 example session, each shown as in Fig. 1F-H -
single-trial raster, PSTH with running speed, and running vs stationary depth
tuning curve.

Panels **J-L**: population summaries - the distribution of the proportion of
depth-tuned neurons across sessions and per mouse, and the preferred depth
distribution for the 5-depth and 8-depth session cohorts.

In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, mannwhitneyu

import flexiznam as flz
from cottage_analysis.analysis import spheres, common_utils
from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.plotting import depth_selectivity_plots, plotting_utils
from cottage_analysis.plotting import style

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT, rect_cm, panel_letter

style.setup_figure_fonts()

In [ ]:
matplotlib.rcParams["axes.labelpad"] = 1  # x and y axis labels (default 4.0)
matplotlib.rcParams["xtick.major.pad"] = 1  # default 3.5
matplotlib.rcParams["ytick.major.pad"] = 1
matplotlib.rcParams["xtick.minor.pad"] = 1  # default 3.4
matplotlib.rcParams["ytick.minor.pad"] = 1

In [ ]:
project = "hey2_3d-vision_foodres_20220101"
flexilims_session = flz.get_flexilims_session(project)
from v1_depth_map.paths import get_figures_roots

READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# The population panels reuse the combined dataframe cached by
# figure1_depth_selectivity.ipynb (`fig1/neurons_df_all.pickle`); set
# recompute_summary=True there to rebuild it from the per-session pickles.
print(f"Saving output to {SAVE_ROOT}")
if READ_ROOT != SAVE_ROOT:
    print(f"Reading data from {READ_ROOT}")

In [ ]:
# Example session - the same one used for Fig. 1F-I
session_name = "PZAH8.2h_S20230116"

vs_df_example, trials_df_example = spheres.sync_all_recordings(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=project,
    filter_datasets={"anatomical_only": 3, "ast_neuropil": False},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)
suite2p_ds = flz.get_datasets_recursively(
    flexilims_session=flexilims_session,
    origin_name=session_name,
    dataset_type="suite2p_traces",
)
fs = list(suite2p_ds.values())[0][-1].extra_attributes["fs"]

neurons_ds_example = pipeline_utils.create_neurons_ds(
    session_name=session_name,
    flexilims_session=flexilims_session,
    project=None,
    conflicts="skip",
)
neurons_df_example = pd.read_pickle(neurons_ds_example.path_full)
common_utils.add_one_sided_spearman_significance(
    neurons_df_example,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="depth_tuned",
)
print(f"{session_name}: {len(neurons_df_example)} ROIs, {int(neurons_df_example.depth_tuned.sum())} depth-tuned")

In [ ]:
# Population dataframe (all V1 closed-loop sessions), as in Fig. 1J
neurons_df_all = pd.read_pickle(READ_ROOT / "fig1/neurons_df_all.pickle")
neurons_df_all = neurons_df_all[neurons_df_all["iscell"] == 1].copy()
common_utils.add_one_sided_spearman_significance(
    neurons_df_all,
    rval_col="depth_tuning_test_spearmanr_rval_closedloop",
    pval_col="depth_tuning_test_spearmanr_pval_closedloop",
    out_col="depth_tuned",
)
neurons_df_all["mouse"] = neurons_df_all["session"].str.split("_").str[0]

# PZAH6.4b and PZAG3.4f were recorded with 5 virtual depths, every other mouse with 8
FIVE_DEPTH_MICE = ["PZAH6.4b", "PZAG3.4f"]
neurons_df_all["ndepths"] = np.where(
    neurons_df_all["mouse"].isin(FIVE_DEPTH_MICE), 5, 8
)

# Proportion of depth-tuned neurons, one value per session
session_prop = (
    neurons_df_all.groupby(["mouse", "session"])["depth_tuned"].mean().reset_index()
)

print(f"{neurons_df_all.session.nunique()} sessions, {neurons_df_all.mouse.nunique()} mice, {len(neurons_df_all)} neurons")
print(f"Depth-tuned: {int(neurons_df_all.depth_tuned.sum())} ({neurons_df_all.depth_tuned.mean():.3f})")

In [ ]:
# --- Pick the three example neurons -------------------------------------------
# One depth-tuned neuron per depth band (near / middle / far), taking the best
# gaussian fit (highest held-out R^2) among neurons whose fitted amplitude is
# large enough for the single-trial raster to be legible.
def _fit_amp(popt):
    return np.exp(popt[0]) if np.ndim(popt) == 1 and len(popt) == 4 else np.nan


neurons_df_example["fit_amp"] = neurons_df_example[
    "depth_tuning_popt_closedloop_running"
].apply(_fit_amp)
candidates = neurons_df_example[
    (neurons_df_example["depth_tuned"] == 1) & (neurons_df_example["fit_amp"] > 0.4)
].copy()
candidates["preferred_depth_cm"] = (
    candidates["preferred_depth_closedloop_running"] * 100
)

DEPTH_BANDS = {"near": (5, 25), "middle": (25, 150), "far": (150, 700)}
for band, (lo, hi) in DEPTH_BANDS.items():
    in_band = candidates[
        (candidates["preferred_depth_cm"] >= lo) & (candidates["preferred_depth_cm"] < hi)
    ].sort_values("depth_tuning_test_rsq_closedloop", ascending=False)
    print(f"--- {band} ({lo}-{hi} cm), {len(in_band)} candidates")
    print(
        in_band[
            ["roi", "preferred_depth_cm", "fit_amp", "depth_tuning_test_rsq_closedloop"]
        ]
        .head(5)
        .to_string(index=False)
    )

# Top candidate of each band, excluding the ROIs already shown in Fig. 1
# (250 in F-H, and 249 / 319 / 333 / 696 in the four-example grid).
EXAMPLE_ROIS = [261, 638, 742]
EXAMPLE_LABELS = ["Near", "Middle", "Far"]
print()
print(
    neurons_df_example.loc[
        EXAMPLE_ROIS,
        [
            "preferred_depth_closedloop_running",
            "depth_tuning_test_rsq_closedloop",
            "fit_amp",
        ],
    ].to_string()
)

In [ ]:
# ==============================================================================
# Assembled supplementary figure (panels A-L)
# Rows 1-3: three example depth-tuned neurons, laid out as Fig. 1F-H
# Row 4: population summaries of depth tuning
# ==============================================================================
FIG_W, FIG_H = 17.5, 16.0
fig = plt.figure(figsize=(FIG_W * CM, FIG_H * CM))

# Column geometry (cm), shared by the three example rows and matched to Fig. 1F-H
COL_RASTER = (1.15, 5.00)  # (x, width)
COL_PSTH = (8.40, 3.15)
COL_TUNING = (14.00, 3.00)
ROW_H = 2.40  # height of the raster / tuning axes
ROW_BOTTOMS = [12.80, 9.30, 5.80]  # bottom edge of each example row
PSTH_SPEED_H = 0.80  # running speed sub-axes
PSTH_DFF_H = 1.15  # dF/F sub-axes

# Corridor geometry, for the travel-distance scale bar over the raster
CORRIDOR_M = 6.0
BLANK_WINDOW_M = 3.0
NBINS_FULL = 60

example_letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]

for i_row, (roi, band_label, y0) in enumerate(
    zip(EXAMPLE_ROIS, EXAMPLE_LABELS, ROW_BOTTOMS)
):
    pref_cm = neurons_df_example.loc[roi, "preferred_depth_closedloop_running"] * 100
    is_last_row = i_row == len(EXAMPLE_ROIS) - 1

    # ---- Raster: single trials at every depth (as Fig. 1F) --------------------
    panel_letter(fig, example_letters[3 * i_row], COL_RASTER[0] - 1.0, y0 + ROW_H + 0.55)
    _, ax_raster = depth_selectivity_plots.plot_raster_all_depths(
        trials_df=trials_df_example,
        roi=roi,
        is_closed_loop=True,
        corridor_length=CORRIDOR_M,
        blank_length=BLANK_WINDOW_M,
        nbins=NBINS_FULL,
        vmax=2,
        plot=True,
        cbar_width=0.01,
        fontsize_dict=FONTSIZE_DICT,
        position=tuple(rect_cm(fig, COL_RASTER[0], y0, COL_RASTER[1], ROW_H)),
    )
    ax_raster.set_title(
        f"{band_label} - cell {i_row + 1}, preferred depth {pref_cm:.0f} cm",
        fontsize=FONTSIZE_DICT["title"],
        loc="left",
        pad=13 if i_row == 0 else 2,  # clear the scale bar on the first row
    )
    if not is_last_row:
        ax_raster.set_xlabel("")

    if i_row == 0:
        # Travel-distance scale bar spanning the corridor part of the first depth block
        m_per_bin = (CORRIDOR_M + 2 * BLANK_WINDOW_M) / NBINS_FULL
        x0_bar = BLANK_WINDOW_M / m_per_bin - 0.5
        x1_bar = x0_bar + CORRIDOR_M / m_per_bin
        ax_raster.plot(
            [x0_bar, x1_bar],
            [1.04, 1.04],
            transform=ax_raster.get_xaxis_transform(),
            color="k",
            linewidth=2,
            solid_capstyle="butt",
            clip_on=False,
        )
        ax_raster.text(
            x1_bar + 12,
            1.04,
            f"{CORRIDOR_M:g} m of travel",
            transform=ax_raster.get_xaxis_transform(),
            ha="left",
            va="center",
            fontsize=FONTSIZE_DICT["legend"],
        )

    # ---- PSTH + running speed (as Fig. 1G) -----------------------------------
    panel_letter(
        fig, example_letters[3 * i_row + 1], COL_PSTH[0] - 1.0, y0 + ROW_H + 0.55
    )
    ax_psth = fig.add_axes(
        rect_cm(
            fig, COL_PSTH[0], y0 + ROW_H - PSTH_DFF_H, COL_PSTH[1], PSTH_DFF_H
        )
    )
    depth_selectivity_plots.plot_PSTH(
        trials_df=trials_df_example,
        roi=roi,
        is_closed_loop=True,
        use_col="dff",
        corridor_length=CORRIDOR_M,
        blank_length=BLANK_WINDOW_M,
        nbins=NBINS_FULL,
        frame_rate=fs,
        fontsize_dict=FONTSIZE_DICT,
        linewidth=1,
        legend_on=(i_row == 0),
        show_ci=False,
        ylim=(0, None),
    )
    # plot_PSTH's auto limit gives 3-decimal tick labels; round it to 1 decimal
    dff_max = common_utils.ceil(ax_psth.get_ylim()[1], base=5)
    ax_psth.set_ylim(0, dff_max)
    ax_psth.set_yticks([0, dff_max])
    ax_psth.set_xticks([])
    ax_psth.set_xlabel("")

    ax_speed = fig.add_axes(
        rect_cm(fig, COL_PSTH[0], y0, COL_PSTH[1], PSTH_SPEED_H)
    )
    depth_selectivity_plots.plot_PSTH(
        trials_df=trials_df_example,
        roi=roi,
        is_closed_loop=True,
        use_col="RS",
        corridor_length=CORRIDOR_M,
        blank_length=BLANK_WINDOW_M,
        nbins=NBINS_FULL,
        frame_rate=fs,
        fontsize_dict=FONTSIZE_DICT,
        linewidth=1,
        legend_on=False,
        show_ci=False,
        ylim=(0, 115),
    )
    ax_speed.set_ylabel("Running\nspeed (cm/s)", fontsize=FONTSIZE_DICT["label"])
    if not is_last_row:
        ax_speed.set_xlabel("")

    # ---- Depth tuning, running vs stationary (as Fig. 1H) --------------------
    panel_letter(
        fig, example_letters[3 * i_row + 2], COL_TUNING[0] - 1.0, y0 + ROW_H + 0.55
    )
    ax_tuning = fig.add_axes(
        rect_cm(fig, COL_TUNING[0], y0, COL_TUNING[1], ROW_H)
    )
    depth_selectivity_plots.plot_running_stationary_depth_tuning(
        roi=roi,
        roi_num=i_row + 1,
        i=i_row,  # i == 0 draws the legend, i % 3 == 2 keeps the depth tick labels
        ax=ax_tuning,
        neurons_df=neurons_df_example,
        trials_df=trials_df_example,
        depth_tuning_kwargs=dict(
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1.5,
            linecolor="royalblue",
            closed_loop=1,
            fontsize_dict=FONTSIZE_DICT,
            markersize=6,
            markeredgecolor="w",
        ),
        fontsize_dict=FONTSIZE_DICT,
        legend_loc="upper right",  # "Cell N" already sits in the upper left
    )
    ax_tuning.set_ylabel(r"$\Delta$F/F", fontsize=FONTSIZE_DICT["label"])
    if is_last_row:
        ax_tuning.set_xlabel("Virtual depth (cm)", fontsize=FONTSIZE_DICT["label"])
    else:
        ax_tuning.set_xlabel("")
        ax_tuning.set_xticklabels([])

# ==============================================================================
# Row 4: population summaries
# ==============================================================================
POP_Y, POP_H = 1.30, 2.30

# ---- Panel J: proportion of depth-tuned neurons across sessions --------------
panel_letter(fig, "J", 0.15, POP_Y + POP_H + 0.75)
ax_j = fig.add_axes(rect_cm(fig, 1.15, POP_Y, 3.60, POP_H))
plt.sca(ax_j)
depth_selectivity_plots.plot_depth_neuron_perc_hist(
    results_df=neurons_df_all,
    use_col="depth_tuned",
    bins=np.arange(0, 1.0, 0.05),
    markersize=5,
    fontsize_dict=FONTSIZE_DICT,
)
ax_j.set_xlabel("Proportion of\ndepth-tuned neurons", fontsize=FONTSIZE_DICT["label"])

# ---- Panels K, L: preferred depth distribution, 5- vs 8-depth sessions -------
tuned = neurons_df_all[neurons_df_all["depth_tuned"] == 1]
# Virtual depths presented by each cohort's protocol, in cm (5-depth mice saw
# geomspace(0.06, 6, 5) m, 8-depth mice geomspace(0.05, 6.4, 8) m)
PRESENTED_DEPTHS_CM = {5: np.geomspace(6, 600, 5), 8: np.geomspace(5, 640, 8)}

cohort_axes = []
for letter, x_cm, ndepths in [("K", 6.60, 5), ("L", 12.60, 8)]:
    panel_letter(fig, letter, x_cm - 1.05, POP_Y + POP_H + 0.75)
    ax_l = fig.add_axes(rect_cm(fig, x_cm, POP_Y, 4.20, POP_H))
    plt.sca(ax_l)
    cohort = tuned[tuned["ndepths"] == ndepths]
    depth_selectivity_plots.plot_preferred_depth_hist(
        results_df=cohort,
        use_col="preferred_depth_closedloop",
        nbins=20,
        fontsize_dict=FONTSIZE_DICT,
    )
    # Mark the depths that were actually presented (the x axis is log depth)
    for depth_cm in PRESENTED_DEPTHS_CM[ndepths]:
        ax_l.axvline(
            np.log(depth_cm),
            color="dimgray",
            linestyle="--",
            linewidth=0.5,
            dashes=(3, 2),
            zorder=5,
        )
    ax_l.set_title(
        f"{ndepths} virtual depths", fontsize=FONTSIZE_DICT["title"], pad=2
    )
    ax_l.set_xlabel("Preferred virtual\ndepth (cm)", fontsize=FONTSIZE_DICT["label"])
    if letter == "L":
        ax_l.set_ylabel("")
    cohort_axes.append(ax_l)

# Share the y axis between the two cohorts, so the wider spread of the 8-depth
# sessions can be read off directly against the 5-depth ones
cohort_ymax = max(ax.get_ylim()[1] for ax in cohort_axes)
for ax in cohort_axes:
    ax.set_ylim(0, cohort_ymax)
    ax.set_yticks([0, cohort_ymax])

out_svg = SAVE_ROOT / "figsupp3_depth_pop.svg"
style.savefig(out_svg, fig=fig)
print(f"Saved {out_svg}")

# Stats

In [ ]:
# Numbers quoted in the legend of the population panels
print("--- Panel J: proportion of depth-tuned neurons per session")
print(f"{len(session_prop)} sessions from {session_prop.mouse.nunique()} mice")
print(
    f"median {session_prop.depth_tuned.median():.3f}, "
    f"range {session_prop.depth_tuned.min():.3f} - {session_prop.depth_tuned.max():.3f}"
)

print()
print("--- Panels K, L: preferred depth, 5-depth vs 8-depth sessions")
pref_by_cohort = {}
for ndepths in [5, 8]:
    cohort = tuned[tuned["ndepths"] == ndepths]
    pref_cm = cohort["preferred_depth_closedloop"].values * 100
    pref_cm = pref_cm[np.isfinite(pref_cm)]
    pref_by_cohort[ndepths] = pref_cm
    print(
        f"{ndepths}-depth: {cohort.session.nunique()} sessions, "
        f"{cohort.mouse.nunique()} mice, {len(cohort)} depth-tuned neurons, "
        f"median preferred depth {np.median(pref_cm):.1f} cm "
        f"(IQR {np.percentile(pref_cm, 25):.1f} - {np.percentile(pref_cm, 75):.1f} cm)"
    )
ks = ks_2samp(np.log(pref_by_cohort[5]), np.log(pref_by_cohort[8]))
mw = mannwhitneyu(pref_by_cohort[5], pref_by_cohort[8])
print(f"KS test on log preferred depth: D = {ks.statistic:.3f}, p = {ks.pvalue:.2e}")
print(f"Mann-Whitney U: U = {mw.statistic:.0f}, p = {mw.pvalue:.2e}")